# Case Assignment #1
## Sarthak Jain, MSIS 510 B

In [214]:
# Add imports
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf


In [215]:
# Load the datasets
df_train = pd.read_csv("train_data-1.csv")
df_test  = pd.read_csv("test_data-1.csv")
df_site_const = pd.read_csv("site_const_data-1-1.csv")

In [216]:
# Rename columns to replace '.' with '_' to use in smf methods
for df in (df_train, df_test, df_site_const):   # use your own dataframe names
    df.columns = df.columns.str.replace('.', '_', regex=False)

### Q1

In [217]:
testing_size = df_test.shape[0]
training_size = df_train.shape[0] 
print("Testing Set Size:", testing_size)
print("Training Set Size:", training_size)

total_sample_data = testing_size + training_size
print("Total Sample Size:", total_sample_data)

training_percentage = 100 * (training_size/total_sample_data)
testing_percentage =  100 * (testing_size/total_sample_data)

print("Training %:", training_percentage)
print("Testing %:", testing_percentage)


Testing Set Size: 85
Training Set Size: 374
Total Sample Size: 459
Training %: 81.48148148148148
Testing %: 18.51851851851852


### Q2

In [218]:
# Linear Regression
linear_regression = smf.ols('annual_profit ~ agg_inc + sqft + col_grad + com60', data=df_train)
linear_regression_results = linear_regression.fit()

# Print model summary
print(linear_regression_results.summary())

                            OLS Regression Results                            
Dep. Variable:          annual_profit   R-squared:                       0.786
Model:                            OLS   Adj. R-squared:                  0.784
Method:                 Least Squares   F-statistic:                     339.1
Date:                Tue, 03 Nov 2026   Prob (F-statistic):          3.73e-122
Time:                        20:35:46   Log-Likelihood:                -5107.5
No. Observations:                 374   AIC:                         1.022e+04
Df Residuals:                     369   BIC:                         1.024e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    8.36e+04   4.11e+04      2.035      0.0

In [219]:
print(linear_regression_results.params.apply(lambda x: f'{x:,.6f}'))

Intercept     83,597.092456
agg_inc            0.002807
sqft             383.363054
col_grad     346,821.138695
com60        218,265.764624
dtype: str


### Q3

In [220]:
milagro_store_scenario = {
    'agg_inc': 100000000,
    'sqft': 800,
    'col_grad': 0.3,
    'com60': 0.1
}
annual_profitability = linear_regression_results.predict(milagro_store_scenario)
# print(annual_profitability)
print(f"Predicted profitability is: ${annual_profitability[0]:,.2f}")

Predicted profitability is: $796,894.99


### Q4

In [221]:
# The R^2 for the training data. Need SSE and SST in order to calculate R^2
# a) R-squared value on training data
train_predictions = linear_regression_results.predict(df_train)
train_actual = df_train['annual_profit']

# Calculate SSE and SST for training data
train_mean = train_actual.mean()
train_sst = ((train_actual - train_mean) ** 2).sum()
train_sse = ((train_actual - train_predictions) ** 2).sum()

# Calculate R-squared for training data
train_r2 = 1 - (train_sse / train_sst)
print("Training data:")
print(f"R-squared: {train_r2:.4f}")


# The R^2 for the testing data. Need SSE and SST in order to calculate R^2
test_predictions = linear_regression_results.predict(df_test)
test_actual = df_test['annual_profit']

# Calculate SST and SSE for test data
test_sst = ((test_actual - train_mean) ** 2).sum()
test_sse = ((test_actual - test_predictions) ** 2).sum()

# Calculate R-squared for test data
test_r2 = 1 - (test_sse / test_sst)
print("Test data:")
print(f"R-squared: {test_r2:.4f}")

Training data:
R-squared: 0.7861
Test data:
R-squared: 0.7201


### Q5

In [222]:
p_values = linear_regression_results.pvalues
print(p_values)

for feature, p_value in p_values.items():
    if p_value < 0.05:
        print(f"The feature '{feature}' is statistically significant (p-value: {p_value:.4f})")
    else:
        print(f"The feature '{feature}' is NOT statistically significant (p-value: {p_value:.4f})")

#Smallest p-value
smallest_p_value_feature = p_values.idxmin()
print(f"The feature with the smallest p-value is '{smallest_p_value_feature}' (p-value: {p_values[smallest_p_value_feature]:.4f})")
#Largest p-value
largest_p_value_feature = p_values.idxmax()
print(f"The feature with the largest p-value is '{largest_p_value_feature}' (p-value: {p_values[largest_p_value_feature]:.4f})")

Intercept    4.258413e-02
agg_inc      5.249611e-62
sqft         1.063204e-09
col_grad     2.309665e-03
com60        2.686647e-02
dtype: float64
The feature 'Intercept' is statistically significant (p-value: 0.0426)
The feature 'agg_inc' is statistically significant (p-value: 0.0000)
The feature 'sqft' is statistically significant (p-value: 0.0000)
The feature 'col_grad' is statistically significant (p-value: 0.0023)
The feature 'com60' is statistically significant (p-value: 0.0269)
The feature with the smallest p-value is 'agg_inc' (p-value: 0.0000)
The feature with the largest p-value is 'Intercept' (p-value: 0.0426)


### Q6

In [223]:
exclude = ['state', 'store_number', 'annual_profit']
include = [col for col in df_train.columns if col not in exclude]

print("Features included in the model:")
for feature in include:
    print(f"  - {feature}")

print("Numerical predictor variables included in the model:")
for feature in include:
    if df_train[feature].dtype in ['int64', 'float64']:
        print(f"  - {feature}")




Features included in the model:
  - agg_inc
  - sqft
  - col_grad
  - com60
  - lci
  - nearcomp
  - nearmil
  - freestand
  - gini
  - housemed
Numerical predictor variables included in the model:
  - agg_inc
  - sqft
  - col_grad
  - com60
  - lci
  - nearcomp
  - nearmil
  - freestand
  - gini
  - housemed


In [224]:
correlation_matrix = df_train[include].corr()
print("Correlation matrix:")
print(correlation_matrix.round(5))


Correlation matrix:
           agg_inc     sqft  col_grad    com60      lci  nearcomp  nearmil  \
agg_inc    1.00000  0.51183   0.67019 -0.23883 -0.31483  -0.14758  0.16018   
sqft       0.51183  1.00000   0.35303 -0.05535 -0.29966  -0.07415  0.12176   
col_grad   0.67019  0.35303   1.00000 -0.22387 -0.31355  -0.18182  0.06533   
com60     -0.23883 -0.05535  -0.22387  1.00000 -0.00408   0.02477  0.03771   
lci       -0.31483 -0.29966  -0.31355 -0.00408  1.00000   0.07143 -0.11476   
nearcomp  -0.14758 -0.07415  -0.18182  0.02477  0.07143   1.00000 -0.11045   
nearmil    0.16018  0.12176   0.06533  0.03771 -0.11476  -0.11045  1.00000   
freestand  0.17311  0.15047   0.16615 -0.03201 -0.21953   0.10422  0.21137   
gini       0.06861  0.11095   0.00135 -0.03507 -0.10490   0.11227 -0.01217   
housemed   0.48639  0.17686   0.54693 -0.10569 -0.15198  -0.17459  0.03186   

           freestand     gini  housemed  
agg_inc      0.17311  0.06861   0.48639  
sqft         0.15047  0.11095   0.176

In [225]:
#The three pairs of variables with the strongest correlations (highest absolute values)
strongest_correlations = []

for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        var1 = correlation_matrix.columns[i]
        var2 = correlation_matrix.columns[j]
        corr_value = correlation_matrix.iloc[i, j]
        strongest_correlations.append((var1, var2, abs(corr_value)))

# Sort by absolute correlation value in descending order and take the top 3
strongest_correlations.sort(key=lambda x: x[2], reverse=True)
top_3 = strongest_correlations[:3]

print("Three pairs of variables with the strongest correlations:")
for var1, var2, corr_value in top_3:
    print(f"  - {var1} and {var2}: {corr_value:.5f}")

Three pairs of variables with the strongest correlations:
  - agg_inc and col_grad: 0.67019
  - col_grad and housemed: 0.54693
  - agg_inc and sqft: 0.51183


### Q7

In [226]:
complete_regression = smf.ols('annual_profit ~  agg_inc + col_grad + com60 + sqft + lci + nearcomp + nearmil + freestand + gini + housemed', data=df_train).fit()
print(complete_regression.summary())

                            OLS Regression Results                            
Dep. Variable:          annual_profit   R-squared:                       0.918
Model:                            OLS   Adj. R-squared:                  0.916
Method:                 Least Squares   F-statistic:                     406.1
Date:                Tue, 03 Nov 2026   Prob (F-statistic):          2.74e-190
Time:                        20:35:46   Log-Likelihood:                -4928.3
No. Observations:                 374   AIC:                             9879.
Df Residuals:                     363   BIC:                             9922.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   1.279e+05    6.2e+04      2.062      0.0

In [227]:
complete_p_values = complete_regression.pvalues
print(complete_p_values)
for feature, p_value in complete_p_values.items():
    if p_value < 0.05:
        print(f"The feature '{feature}' is statistically significant (p-value: {p_value:.4f})")
    else:
        print(f"The feature '{feature}' is NOT statistically significant (p-value: {p_value:.4f})")

Intercept     3.989194e-02
agg_inc      9.111761e-100
col_grad      1.340048e-05
com60         3.372966e-03
sqft          3.808832e-13
lci           4.380442e-04
nearcomp      1.182668e-18
nearmil       3.188841e-06
freestand     7.175235e-51
gini          7.237427e-01
housemed      3.430107e-01
dtype: float64
The feature 'Intercept' is statistically significant (p-value: 0.0399)
The feature 'agg_inc' is statistically significant (p-value: 0.0000)
The feature 'col_grad' is statistically significant (p-value: 0.0000)
The feature 'com60' is statistically significant (p-value: 0.0034)
The feature 'sqft' is statistically significant (p-value: 0.0000)
The feature 'lci' is statistically significant (p-value: 0.0004)
The feature 'nearcomp' is statistically significant (p-value: 0.0000)
The feature 'nearmil' is statistically significant (p-value: 0.0000)
The feature 'freestand' is statistically significant (p-value: 0.0000)
The feature 'gini' is NOT statistically significant (p-value: 0.7237)


### Q8

#### Model A: Kathleen's Original Model

In [228]:
model_a = smf.ols('annual_profit ~ agg_inc + sqft + col_grad + com60', data=df_train).fit()
print(model_a.summary())

                            OLS Regression Results                            
Dep. Variable:          annual_profit   R-squared:                       0.786
Model:                            OLS   Adj. R-squared:                  0.784
Method:                 Least Squares   F-statistic:                     339.1
Date:                Tue, 03 Nov 2026   Prob (F-statistic):          3.73e-122
Time:                        20:35:46   Log-Likelihood:                -5107.5
No. Observations:                 374   AIC:                         1.022e+04
Df Residuals:                     369   BIC:                         1.024e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    8.36e+04   4.11e+04      2.035      0.0

In [229]:
model_a_train_r2 = model_a.rsquared

model_a_test_predictions = model_a.predict(df_test)
model_a_test_actual = df_test['annual_profit']
model_a_train_actual = df_train['annual_profit']
model_a_train_mean = model_a_train_actual.mean()
    
model_a_test_sst = ((model_a_test_actual - model_a_train_mean) ** 2).sum()
model_a_test_sse = ((model_a_test_actual - model_a_test_predictions) ** 2).sum()
model_a_test_r2 = 1 - (model_a_test_sse / model_a_test_sst)
    
    # Total predicted profitability for 48 construction sites
model_a_site_predictions = model_a.predict(df_site_const)
total_profit_millions_model_a = model_a_site_predictions.sum() / 1000000

print(f"Model A R-squared (training data): {model_a_train_r2:.4f}")
print(f"Model A R-squared (testing data): {model_a_test_r2:.4f}")
print(f"Total predicted profitability for 48 construction sites (Model A): ${total_profit_millions_model_a:,.2f} million")

Model A R-squared (training data): 0.7861
Model A R-squared (testing data): 0.7201
Total predicted profitability for 48 construction sites (Model A): $40.02 million


#### Model B: Full Model

In [230]:
model_b = smf.ols('annual_profit ~ agg_inc + col_grad + com60 + sqft + lci + nearcomp + nearmil + freestand + gini + housemed', data=df_train).fit()
print(model_b.summary())

                            OLS Regression Results                            
Dep. Variable:          annual_profit   R-squared:                       0.918
Model:                            OLS   Adj. R-squared:                  0.916
Method:                 Least Squares   F-statistic:                     406.1
Date:                Tue, 03 Nov 2026   Prob (F-statistic):          2.74e-190
Time:                        20:35:47   Log-Likelihood:                -4928.3
No. Observations:                 374   AIC:                             9879.
Df Residuals:                     363   BIC:                             9922.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   1.279e+05    6.2e+04      2.062      0.0

In [231]:
model_b_train_r2 = model_b.rsquared

model_b_test_predictions = model_b.predict(df_test)
model_b_test_actual = df_test['annual_profit']
model_b_train_actual = df_train['annual_profit']
model_b_train_mean = model_b_train_actual.mean()
    
model_b_test_sst = ((model_b_test_actual - model_b_train_mean) ** 2).sum()
model_b_test_sse = ((model_b_test_actual - model_b_test_predictions) ** 2).sum()
model_b_test_r2 = 1 - (model_b_test_sse / model_b_test_sst)
    
    # Total predicted profitability for 48 construction sites
model_b_site_predictions = model_b.predict(df_site_const)
total_profit_millions_model_b = model_b_site_predictions.sum() / 1000000

print(f"Model B R-squared (training data): {model_b_train_r2:.4f}")
print(f"Model B R-squared (testing data): {model_b_test_r2:.4f}")
print(f"Total predicted profitability for 48 construction sites (Model B): ${total_profit_millions_model_b:,.2f} million")

Model B R-squared (training data): 0.9179
Model B R-squared (testing data): 0.8272
Total predicted profitability for 48 construction sites (Model B): $36.06 million


#### Model C: Parsimonious Model

In [232]:
model_c = smf.ols('annual_profit ~ agg_inc + col_grad + com60 + sqft + lci + nearcomp + nearmil + freestand + gini + housemed', data=df_train).fit()
print(model_c.summary())


# Non significant variables (p >= 0.05) from Model C
p_values_c = model_c.pvalues[1:]
non_significant = p_values_c[p_values_c >= 0.05].index.tolist()
print(f"\nNon-significant variables (p >= 0.05): {non_significant}")

# Identify highly correlated pairs (|correlation| > 0.70)
all_vars = ['agg_inc', 'sqft', 'col_grad', 'com60', 'lci', 'nearcomp', 'nearmil', 'freestand', 'gini', 'housemed']
corr_with_target = df_train[all_vars + ['annual_profit']].corr()['annual_profit'][all_vars]

print(f"\nCorrelations with annual_profit:")
for var in all_vars:
    print(f"  {var:12s}: {corr_with_target[var]:.4f}")

# Find pairs with |correlation| > 0.70
high_corr_pairs = []
for i, var1 in enumerate(all_vars):
    for var2 in all_vars[i+1:]:
        corr_val = df_train[[var1, var2]].corr().iloc[0, 1]
        if abs(corr_val) > 0.70:
            high_corr_pairs.append((var1, var2, corr_val))
            print(f"\nHigh correlation found: {var1} & {var2} = {corr_val:.4f}")
            print(f"  {var1} correlation with target: {corr_with_target[var1]:.4f}")
            print(f"  {var2} correlation with target: {corr_with_target[var2]:.4f}")
            
# Variables to remove
vars_to_remove = set(non_significant)

# For high correlation pairs, remove the one with weaker correlation to target
for var1, var2, _ in high_corr_pairs:
    if abs(corr_with_target[var1]) > abs(corr_with_target[var2]):
        vars_to_remove.add(var2)
        print(f"  → Removing {var2} (weaker correlation with target)")
    else:
        vars_to_remove.add(var1)
        print(f"  → Removing {var1} (weaker correlation with target)")

# Variables to keep
vars_to_keep = [v for v in all_vars if v not in vars_to_remove]
print(f"\nVariables removed: {sorted(vars_to_remove)}")
print(f"Variables kept in Model C: {vars_to_keep}")

# Build Model C
formula_c = 'annual_profit ~ ' + ' + '.join(vars_to_keep)
print(f"\nFormula: {formula_c}")

model_c = smf.ols(formula_c, data=df_train).fit()
print(model_c.summary())

# Evaluate Model C
model_c_train_r2 = model_c.rsquared

model_c_test_predictions = model_c.predict(df_test)
model_c_test_actual = df_test['annual_profit']
model_c_train_actual = df_train['annual_profit']
model_c_train_mean = model_c_train_actual.mean()
    
model_c_test_sst = ((model_c_test_actual - model_c_train_mean) ** 2).sum()
model_c_test_sse = ((model_c_test_actual - model_c_test_predictions) ** 2).sum()
model_c_test_r2 = 1 - (model_c_test_sse / model_c_test_sst)
    
    # Total predicted profitability for 48 construction sites
model_c_site_predictions = model_c.predict(df_site_const)
total_profit_millions_model_c = model_c_site_predictions.sum() / 1000000

print(f"Model C R-squared (training data): {model_c_train_r2:.4f}")
print(f"Model C R-squared (testing data): {model_c_test_r2:.4f}")
print(f"Total predicted profitability for 48 construction sites (Model C): ${total_profit_millions_model_c:,.2f} million")

                            OLS Regression Results                            
Dep. Variable:          annual_profit   R-squared:                       0.918
Model:                            OLS   Adj. R-squared:                  0.916
Method:                 Least Squares   F-statistic:                     406.1
Date:                Tue, 03 Nov 2026   Prob (F-statistic):          2.74e-190
Time:                        20:35:47   Log-Likelihood:                -4928.3
No. Observations:                 374   AIC:                             9879.
Df Residuals:                     363   BIC:                             9922.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   1.279e+05    6.2e+04      2.062      0.0

#### Model D: Alternative Model

In [233]:
model_d_best_option = ''
model_d_highest_profit = 0
model_d_highest_r2 = 0

##### Using lci 

In [239]:
model_d_lci = smf.ols('annual_profit ~ agg_inc + col_grad + com60 + sqft + lci', data=df_train).fit()
print(model_d_lci.summary())

test_model_d_lci_pred = model_d_lci.predict(df_test)
test_model_d_lci_actual = df_test['annual_profit']
train_model_d_lci_mean = df_train['annual_profit'].mean()
test_model_d_lci_sst = ((test_model_d_lci_actual - train_model_d_lci_mean) ** 2).sum()
test_model_d_lci_sse = ((test_model_d_lci_actual - test_model_d_lci_pred) ** 2).sum()
test_model_d_lci_r2 = 1 - (test_model_d_lci_sse / test_model_d_lci_sst)

model_d_lci_site_pred = model_d_lci.predict(df_site_const)
model_d_lci_profit = model_d_lci_site_pred.sum() / 1000000

print(f"\nModel D LCI R-squared (testing data): {test_model_d_lci_r2:.4f}")
print(f"\nModel D LCI R-squared (training data): {model_d_lci.rsquared:.4f}")
print(f"Model D LCI Predicted Profitability: ${model_d_lci_profit:,.2f} million")

if test_model_d_lci_r2 > model_d_highest_r2:
    
    if model_d_lci_profit > model_d_highest_profit and model_d_lci_profit >= 40:
        model_d_best_option = 'lci'
        model_d_highest_r2 = test_model_d_lci_r2
        model_d_highest_profit = model_d_lci_profit

                            OLS Regression Results                            
Dep. Variable:          annual_profit   R-squared:                       0.797
Model:                            OLS   Adj. R-squared:                  0.794
Method:                 Least Squares   F-statistic:                     288.6
Date:                Tue, 03 Nov 2026   Prob (F-statistic):          6.06e-125
Time:                        20:37:12   Log-Likelihood:                -5097.9
No. Observations:                 374   AIC:                         1.021e+04
Df Residuals:                     368   BIC:                         1.023e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   3.905e+05   8.05e+04      4.851      0.0

##### Using nearcomp 

In [235]:
model_d_nearcomp = smf.ols('annual_profit ~ agg_inc + col_grad + com60 + sqft + nearcomp', data=df_train).fit()
print(model_d_nearcomp.summary())

test_model_d_nearcomp_pred = model_d_nearcomp.predict(df_test)
test_model_d_nearcomp_actual = df_test['annual_profit']
train_model_d_nearcomp_mean = df_train['annual_profit'].mean()
test_model_d_nearcomp_sst = ((test_model_d_nearcomp_actual - train_model_d_nearcomp_mean) ** 2).sum()
test_model_d_nearcomp_sse = ((test_model_d_nearcomp_actual - test_model_d_nearcomp_pred) ** 2).sum()
test_model_d_nearcomp_r2 = 1 - (test_model_d_nearcomp_sse / test_model_d_nearcomp_sst)

model_d_nearcomp_site_pred = model_d_nearcomp.predict(df_site_const)
model_d_nearcomp_profit = model_d_nearcomp_site_pred.sum() / 1000000

print(f"\nModel D Nearcomp R-squared (testing data): {test_model_d_nearcomp_r2:.4f}")
print(f"Model D Nearcomp Predicted Profitability: ${model_d_nearcomp_profit:,.2f} million")

if test_model_d_nearcomp_r2 > model_d_highest_r2:
    
    if model_d_nearcomp_profit > model_d_highest_profit and model_d_nearcomp_profit >= 40:
        model_d_best_option = 'nearcomp'
        model_d_highest_r2 = test_model_d_nearcomp_r2
        model_d_highest_profit = model_d_nearcomp_profit

                            OLS Regression Results                            
Dep. Variable:          annual_profit   R-squared:                       0.818
Model:                            OLS   Adj. R-squared:                  0.815
Method:                 Least Squares   F-statistic:                     330.6
Date:                Tue, 03 Nov 2026   Prob (F-statistic):          1.09e-133
Time:                        20:35:47   Log-Likelihood:                -5077.4
No. Observations:                 374   AIC:                         1.017e+04
Df Residuals:                     368   BIC:                         1.019e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -4.46e+04   4.12e+04     -1.083      0.2

##### Using nearmil 

In [236]:
model_d_nearmil = smf.ols('annual_profit ~ agg_inc + col_grad + com60 + sqft + nearmil', data=df_train).fit()
print(model_d_nearmil.summary())

test_model_d_nearmil_pred = model_d_nearmil.predict(df_test)
test_model_d_nearmil_actual = df_test['annual_profit']
train_model_d_nearmil_mean = df_train['annual_profit'].mean()
test_model_d_nearmil_sst = ((test_model_d_nearmil_actual - train_model_d_nearmil_mean) ** 2).sum()
test_model_d_nearmil_sse = ((test_model_d_nearmil_actual - test_model_d_nearmil_pred) ** 2).sum()
test_model_d_nearmil_r2 = 1 - (test_model_d_nearmil_sse / test_model_d_nearmil_sst)

model_d_nearmil_site_pred = model_d_nearmil.predict(df_site_const)
model_d_nearmil_profit = model_d_nearmil_site_pred.sum() / 1000000

print(f"\nModel D Nearmil R-squared (testing data): {test_model_d_nearmil_r2:.4f}")
print(f"Model D Nearmil Predicted Profitability: ${model_d_nearmil_profit:,.2f} million")

if test_model_d_nearmil_r2 > model_d_highest_r2:
    
    if model_d_nearmil_profit > model_d_highest_profit and model_d_nearmil_profit >= 40:
        model_d_best_option = 'nearmil'
        model_d_highest_r2 = test_model_d_nearmil_r2
        model_d_highest_profit = model_d_nearmil_profit

                            OLS Regression Results                            
Dep. Variable:          annual_profit   R-squared:                       0.800
Model:                            OLS   Adj. R-squared:                  0.797
Method:                 Least Squares   F-statistic:                     293.9
Date:                Tue, 03 Nov 2026   Prob (F-statistic):          4.22e-126
Time:                        20:35:47   Log-Likelihood:                -5095.2
No. Observations:                 374   AIC:                         1.020e+04
Df Residuals:                     368   BIC:                         1.023e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   3.773e+04   4.09e+04      0.924      0.3

##### Using freestand

In [237]:
model_d_freestand = smf.ols('annual_profit ~ agg_inc + col_grad + com60 + sqft + freestand', data=df_train).fit()
print(model_d_freestand.summary())

test_model_d_freestand_pred = model_d_freestand.predict(df_test)
test_model_d_freestand_actual = df_test['annual_profit']
train_model_d_freestand_mean = df_train['annual_profit'].mean()
test_model_d_freestand_sst = ((test_model_d_freestand_actual - train_model_d_freestand_mean) ** 2).sum()
test_model_d_freestand_sse = ((test_model_d_freestand_actual - test_model_d_freestand_pred) ** 2).sum()
test_model_d_freestand_r2 = 1 - (test_model_d_freestand_sse / test_model_d_freestand_sst)

model_d_freestand_site_pred = model_d_freestand.predict(df_site_const)
model_d_freestand_profit = model_d_freestand_site_pred.sum() / 1000000

print(f"\nModel D Freestand R-squared (testing data): {test_model_d_freestand_r2:.4f}")
print(f"Model D Freestand Predicted Profitability: ${model_d_freestand_profit:,.2f} million")
if test_model_d_freestand_r2 > model_d_highest_r2:
    if model_d_freestand_profit > model_d_highest_profit and model_d_freestand_profit >= 40:
        model_d_best_option = 'freestand'
        model_d_highest_r2 = test_model_d_freestand_r2
        model_d_highest_profit = model_d_freestand_profit

                            OLS Regression Results                            
Dep. Variable:          annual_profit   R-squared:                       0.892
Model:                            OLS   Adj. R-squared:                  0.890
Method:                 Least Squares   F-statistic:                     605.5
Date:                Tue, 03 Nov 2026   Prob (F-statistic):          4.27e-175
Time:                        20:35:47   Log-Likelihood:                -4980.4
No. Observations:                 374   AIC:                             9973.
Df Residuals:                     368   BIC:                             9996.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   9.709e+04   2.93e+04      3.314      0.0

In [238]:
print(f"\nModel D Best Option: {model_d_best_option}")
print(f"Highest R-squared: {model_d_highest_r2:.4f}")
print(f"Highest predicted profitability for 48 construction sites: ${model_d_highest_profit:,.2f} million")


Model D Best Option: lci
Highest R-squared: 0.7670
Highest predicted profitability for 48 construction sites: $40.10 million
